# 01. 봇마당 파일럿 데이터 탐색

수집된 파일럿 데이터(500개 게시글)의 기초 통계와 특성을 파악한다.

In [ ]:
import json
import os
from collections import Counter
from datetime import datetime

# 데이터 로드
DATA_DIR = '../data'

with open(f'{DATA_DIR}/raw/pilot_posts.json', 'r') as f:
    posts = json.load(f)

with open(f'{DATA_DIR}/processed/agent_summary.json', 'r') as f:
    agents = json.load(f)

with open(f'{DATA_DIR}/processed/submadang_summary.json', 'r') as f:
    madangs = json.load(f)

print(f'게시글: {len(posts)}개')
print(f'에이전트: {len(agents)}명')
print(f'마당: {len(madangs)}개')

## 1. 게시글 기초 통계

In [ ]:
# 게시글 길이 분포
content_lengths = [len(p.get('content', '')) for p in posts]
title_lengths = [len(p.get('title', '')) for p in posts]

print('=== 게시글 본문 길이 ===')
print(f'  평균: {sum(content_lengths)/len(content_lengths):.0f}자')
print(f'  중간값: {sorted(content_lengths)[len(content_lengths)//2]}자')
print(f'  최소: {min(content_lengths)}자')
print(f'  최대: {max(content_lengths)}자')

print(f'\n=== 제목 길이 ===')
print(f'  평균: {sum(title_lengths)/len(title_lengths):.0f}자')

# 추천 분포
upvotes = [p.get('upvotes', 0) for p in posts]
print(f'\n=== 추천 수 ===')
print(f'  평균: {sum(upvotes)/len(upvotes):.1f}')
print(f'  최대: {max(upvotes)}')
print(f'  0인 글: {sum(1 for u in upvotes if u == 0)}개 ({sum(1 for u in upvotes if u == 0)/len(upvotes)*100:.0f}%)')

# 댓글 수 분포
comment_counts = [p.get('comment_count', 0) for p in posts]
print(f'\n=== 댓글 수 ===')
print(f'  평균: {sum(comment_counts)/len(comment_counts):.1f}')
print(f'  최대: {max(comment_counts)}')
print(f'  0인 글: {sum(1 for c in comment_counts if c == 0)}개')

## 2. 마당별 분포

In [ ]:
# 마당별 게시글 수
submadang_counts = Counter(p.get('submadang', 'unknown') for p in posts)

print('=== 마당별 게시글 수 ===')
for name, count in submadang_counts.most_common():
    pct = count / len(posts) * 100
    bar = '#' * int(pct / 2)
    print(f'  {name:20} {count:>4}개 ({pct:5.1f}%) {bar}')

## 3. 에이전트 활동 패턴

In [ ]:
# 에이전트별 게시글 수 분포
agent_post_counts = Counter(p.get('author_name', 'unknown') for p in posts)

print('=== 상위 15 에이전트 ===')
for name, count in agent_post_counts.most_common(15):
    pct = count / len(posts) * 100
    bar = '#' * int(pct)
    print(f'  {name:20} {count:>4}개 ({pct:5.1f}%) {bar}')

# 에이전트 집중도
top3 = sum(c for _, c in agent_post_counts.most_common(3))
top5 = sum(c for _, c in agent_post_counts.most_common(5))
print(f'\n상위 3명 점유율: {top3/len(posts)*100:.1f}%')
print(f'상위 5명 점유율: {top5/len(posts)*100:.1f}%')
print(f'총 에이전트 수: {len(agent_post_counts)}명')

## 4. 시계열 패턴

In [ ]:
# 날짜별 게시글 수
date_counts = Counter()
for p in posts:
    created = p.get('created_at', '')
    if created:
        date = created[:10]  # YYYY-MM-DD
        date_counts[date] += 1

print('=== 최근 날짜별 게시글 ===')
for date in sorted(date_counts.keys(), reverse=True)[:14]:
    count = date_counts[date]
    bar = '#' * count
    print(f'  {date} {count:>3}개 {bar}')

## 5. 텍스트 특성 초기 관찰

In [ ]:
import re

# 문체 초기 분석
honorific_patterns = re.compile(r'(습니다|세요|ㅂ니다|주세요|하겠습니다|입니다|합니다)')
casual_patterns = re.compile(r'(ㅋㅋ|ㅎㅎ|ㅠㅠ|진짜|근데|걍|ㄹㅇ)')
question_patterns = re.compile(r'\?')

stats = {
    'honorific_ratio': 0,
    'casual_ratio': 0,
    'question_ratio': 0,
    'emoji_ratio': 0,
}

for p in posts:
    content = p.get('content', '')
    if honorific_patterns.search(content):
        stats['honorific_ratio'] += 1
    if casual_patterns.search(content):
        stats['casual_ratio'] += 1
    if question_patterns.search(content):
        stats['question_ratio'] += 1

n = len(posts)
print('=== 문체 초기 분석 ===')
print(f'  존댓말 포함: {stats["honorific_ratio"]}/{n} ({stats["honorific_ratio"]/n*100:.1f}%)')
print(f'  비격식 표현: {stats["casual_ratio"]}/{n} ({stats["casual_ratio"]/n*100:.1f}%)')
print(f'  질문 포함: {stats["question_ratio"]}/{n} ({stats["question_ratio"]/n*100:.1f}%)')

## 6. 샘플 게시글 확인

In [ ]:
# 다양한 마당의 대표 게시글 보기
shown_madangs = set()
for p in posts:
    m = p.get('submadang', 'unknown')
    if m not in shown_madangs:
        shown_madangs.add(m)
        print(f'\n--- [{m}] {p["author_name"]} ---')
        print(f'제목: {p["title"]}')
        content = p.get('content', '')[:200]
        print(f'내용: {content}...')
        print(f'추천: {p.get("upvotes", 0)} | 댓글: {p.get("comment_count", 0)}')
    if len(shown_madangs) >= 6:
        break